# M28 — Search Meaning with Embeddings

**Objective:** use embeddings to retrieve semantically related items.

Token IDs from M27 are not meanings. The useful whole is a small
retrieval over **bundled** sentence vectors:

`text → vector → cosine against a corpus → ranked neighbors`

Every vector carries a provenance contract: model, version, width,
metric, normalization, pooling. This notebook uses
`v06-teaching-meanpool` version `v06.1`. Nothing is downloaded.
Attention and a search service stay closed (M29, M33).


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a neighbor id, a rank order, a margin, or a
rejected mix.

Do not download an encoder, do not treat a cosine as proof of
equivalence, and do not open an index service. If a failure can be
diagnosed from mismatched metadata, stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M28" / "embedding_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M28.embedding_core import (
    ProvenanceError,
    compare_lexical_and_semantic,
    compatibility_report,
    content_tokens,
    cosine_similarity,
    encode_text,
    inner_product,
    lexical_overlap,
    l2_norm,
    load_canonical_space,
    load_catalog,
    load_encoder,
    load_mismatch_space,
    pairwise_cosine,
    rank_neighbors,
    retrieval_report,
    retrieve_query,
    retrieve_unchecked,
)

canonical = load_canonical_space()
mismatch = load_mismatch_space()
encoder = load_encoder()
CATALOG = load_catalog()

print("repository root:", ROOT)
print("model:", canonical.provenance.model)
print("version:", canonical.provenance.version)
print("metric:", canonical.provenance.metric)
print("normalization:", canonical.provenance.normalization)
print("pooling:", canonical.provenance.pooling)
print("dimensions:", canonical.provenance.dimensions)
print("downloaded:", canonical.provenance.downloaded)
print("documents:", len(canonical.documents()), "queries:", len(canonical.queries()))


## M15 / M27 boundary: vectors and tokens in, attention out

M15 already ranked small vectors with an explicit metric. M27 already
turned text into token IDs. M28 asks what happens when a **sentence**
is a vector you can retrieve with.

What this mission **opens:** document vectors, width, mean pooling, L2,
cosine nearest neighbors, lexical versus semantic disagreement, hard
cases, and provenance compatibility.

What stays **deferred:**
- M29 — attention; queries, keys, values; softmax weights
- M33 — a retrieval service, index, filters, latency

M27 tokenization is not re-taught. The lexical baseline here is Jaccard
on alphanumeric words, a contrast with geometry, not a second tokenizer.


## Frozen teaching fixtures

Declare the useful whole **before** the first score.

| Fixture | Value |
| --- | --- |
| Family | `v06-teaching-embed` |
| Model | `v06-teaching-meanpool` |
| Version | `v06.1` |
| Width | 12 named teaching dimensions |
| Pooling | mean of content tokens |
| Normalization | L2 |
| Metric | cosine |
| Canonical query | `I forgot my password and cannot sign in.` |
| Download | false — JSON in `datasets/M28/` |

Primary sources: `sentence-transformers` and `hf-llm-course` in
`data/source_registry.json`. Skip production encoders here.

The teaching table is authored so paraphrase, lexical traps, and hard
cases are visible. It is not a quality benchmark and not a model-hub
checkpoint.


In [ ]:
print("fingerprint", canonical.provenance.fingerprint())
print("dimension names", canonical.provenance.dimension_names)
print("--- corpus ---")
for item in canonical.documents():
    print(f"{item.id:20s} {item.text}")
print("--- queries ---")
for item in canonical.queries():
    print(f"{item.id:20s} [{item.tags}] {item.text}")
assert canonical.provenance.downloaded is False
assert canonical.provenance.network_required is False
assert canonical.provenance.version == "v06.1"
assert mismatch.provenance.version == "v06.2"


### Identity is part of the contract

Every retrieve below is only valid against a corpus with the same
family, model, version, width, metric, normalization, and pooling.
If any of those change, scores are not comparable. That is the
migration trigger M33 will inherit.

A dimension name such as `account_access` is a teaching label. It is
not a guarantee that a production encoder has an "account neuron."


In [ ]:
for item in (canonical.get("q-password"), canonical.get("d-password-forgot"), canonical.get("d-printer-reset")):
    print(item.id, "shape", (len(item.vector),), "norm", round(l2_norm(item.vector), 6))
    print("  head", tuple(round(value, 4) for value in item.vector[:4]))
report = encoder.encode_report("I forgot my password and cannot sign in.")
print("encode_report", report)
assert abs(l2_norm(canonical.get("d-password-forgot").vector) - 1.0) < 1e-9
assert report["shape"] == (12,)
assert report["downloaded"] is False


### A stored vector is a contract row

Width 12, L2 norm 1, model `v06-teaching-meanpool`, version `v06.1`.
Those facts are data, not decoration. The next cells will score these
rows. Scoring is not yet allowed to mix in `v06.2`.


## Predict before running — nearest items

Timestamp a prediction before `run-rank`.

Query `q-password`: `I forgot my password and cannot sign in.`

The corpus is frozen. Predict the **top three document ids** and
whether a printer document can appear in that top three. Name one
item you expect to lose.

Do not compute cosine yet. A guess from the text is the point.


In [ ]:
password_rank = retrieve_query("q-password")
print(retrieval_report(password_rank))
for item in password_rank.results[:6]:
    print(item.rank, item.id, round(item.score, 4), item.text)
assert password_rank.ids()[:3] == tuple(CATALOG["expected"]["q-password_top3"])
assert "d-printer-reset" not in password_rank.ids()[:3]
print("margin 1-2", password_rank.margin())


### Nearest is a ranking, not a verdict

The password query sits on the account documents. Printer rows exist
in the same corpus and lose. Cosine did not "understand a login
problem." It ranked stored vectors under a declared metric.

Keep the ranking. The next cell looks at a few pairwise scores so the
geometry is visible without a service.


## Predict before running — pairwise cosine

Timestamp a prediction before `run-pairwise`.

Subset: password-forgot, login-reset, printer-reset, approve-refund,
deny-refund, rain.

Predict:
- the diagonal of the cosine matrix
- whether approve-refund and deny-refund are closer than
  password-forgot and printer-reset
- whether the matrix is symmetric

Invariant: the same stored vectors; no new encoding.


In [ ]:
subset_ids = CATALOG["pairwise_subset"]
subset = [canonical.get(item_id) for item_id in subset_ids]
scores, ids = pairwise_cosine(subset)
print("ids", ids)
print(np.round(scores, 3))
approve = canonical.get("d-approve-refund")
deny = canonical.get("d-deny-refund")
password = canonical.get("d-password-forgot")
printer = canonical.get("d-printer-reset")
print("approve vs deny", round(cosine_similarity(approve.vector, deny.vector), 4))
print("password vs printer", round(cosine_similarity(password.vector, printer.vector), 4))
assert abs(float(scores[0, 0]) - 1.0) < 1e-9
assert abs(float(scores[1, 0]) - float(scores[0, 1])) < 1e-9
assert cosine_similarity(approve.vector, deny.vector) > cosine_similarity(password.vector, printer.vector)


### High cosine is cheap to over-read

Approve versus deny is high because the teaching geometry shares refund
mass and only adds a negation axis. That number will return as a hard
case. Password versus printer is low on this table. Neither score is
an explanation of intent.


## Predict before running — paraphrase retrieval

Timestamp a prediction before `run-paraphrase`.

**Change:** replace `q-password` with the paraphrase `q-paraphrase`
(`Please reset my login password.`), which is **not** a corpus copy.

**Invariant:** corpus, model, version, metric, normalization.

Predict whether the top three stay in the account set, how the margin
to the first printer trap moves, and whether the paraphrase can outrank
the original sentence against `d-login-reset`.


In [ ]:
paraphrase_rank = retrieve_query("q-paraphrase")
print("original top", password_rank.ids()[:3], "margin", round(password_rank.margin() or 0.0, 4))
print("paraphrase top", paraphrase_rank.ids()[:3], "margin", round(paraphrase_rank.margin() or 0.0, 4))
print(retrieval_report(paraphrase_rank))
for item in paraphrase_rank.results[:5]:
    print(item.rank, item.id, round(item.score, 4), "lex", round(item.lexical_overlap, 3))
account = {"d-password-forgot", "d-login-reset", "d-cannot-signin"}
assert set(paraphrase_rank.ids()[:3]) <= account
assert paraphrase_rank.top_id == "d-login-reset"
assert "d-printer-queue" not in paraphrase_rank.ids()[:3]
print("printer trap rank", paraphrase_rank.ids().index("d-printer-reset") + 1)


### Paraphrase kept the account neighborhood

The query wording changed and is not a document copy; the corpus and
encoder did not. Nearby account documents stay at the top. Printer-reset
is still available as a lexical trap and still loses under cosine. That
is the operational claim embeddings make: nearby in vector space, not
"the model knows passwords."


## Predict before running — lexical vs semantic

Timestamp a prediction before `run-lexical`.

Two controlled pairs, same corpus:

1. High overlap, different meaning: `Please reset the printer.` versus
   `Please reset the login credentials.`
2. Low overlap, similar meaning: `I cannot access my account.` versus
   `Please reset the login credentials.`

Predict Jaccard versus cosine rank for each. Which ranking puts
`d-login-reset` nearer the printer query?


In [ ]:
printer_query = canonical.get("q-printer")
printer_rows = compare_lexical_and_semantic(
    printer_query.text,
    canonical,
    query_vector=printer_query.vector,
    query_id=printer_query.id,
    query_provenance=canonical.provenance,
)
print("printer query — selected rows")
for row in printer_rows:
    if row["id"] in {"d-printer-reset", "d-printer-queue", "d-login-reset", "d-password-forgot"}:
        print(row)

low_query = canonical.get("q-low-overlap")
low_rows = compare_lexical_and_semantic(
    low_query.text,
    canonical,
    query_vector=low_query.vector,
    query_id=low_query.id,
    query_provenance=canonical.provenance,
)
print("low-overlap query — selected rows")
for row in low_rows:
    if row["id"] in {"d-login-reset", "d-cannot-signin", "d-password-forgot", "d-printer-reset"}:
        print(row)

login_text = canonical.get("d-login-reset").text
printer_lex_login = lexical_overlap(printer_query.text, login_text)
low_lex_login = lexical_overlap(low_query.text, login_text)
print("jaccard printer vs login", round(printer_lex_login, 3))
print("jaccard access-account vs login", round(low_lex_login, 3))
printer_row = next(row for row in printer_rows if row["id"] == "d-printer-reset")
login_row = next(row for row in printer_rows if row["id"] == "d-login-reset")
assert printer_lex_login >= 0.5
assert low_lex_login < 0.25
assert printer_row["semantic_rank"] < login_row["semantic_rank"]
assert login_row["disagrees"]


In [ ]:
xs = [row["lexical_overlap"] for row in printer_rows]
ys = [row["cosine"] for row in printer_rows]
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(xs, ys, c="#1f4e79")
for row in printer_rows:
    if row["id"] in {"d-printer-reset", "d-login-reset", "d-printer-queue", "d-password-forgot"}:
        ax.annotate(row["id"], (row["lexical_overlap"], row["cosine"]), fontsize=8)
ax.set_xlabel("lexical overlap (Jaccard words)")
ax.set_ylabel("cosine similarity")
ax.set_title("Printer query: do overlap and geometry agree?")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
print("login overlap vs cosine", round(login_row["lexical_overlap"], 3), round(login_row["cosine"], 3))
print("printer overlap vs cosine", round(printer_row["lexical_overlap"], 3), round(printer_row["cosine"], 3))


### Overlap is not geometry

`please reset the _` is shared by printer and login, so Jaccard likes
both. Cosine uses the rest of the vector (`printer` vs `login` /
`credentials`) and ranks the device documents first. The access-account
query shares almost no words with login-reset and still sits in the
account neighborhood. Neither ranking is universally correct. They
measure different things.


## Predict before running — negation, number, entity

Timestamp a prediction before `run-hard`.

Three one-change cases, encoder and other text fixed:

1. Negation: `Do not approve the customer refund.` versus the approve
   document that drops only `not`.
2. Number: `fifty` versus `five thousand`.
3. Entity: `ticket 4412` versus `ticket 4413`.

Predict top-1, the hard neighbor, and whether that neighbor's cosine
stays deceptively high (above 0.85) or the margin stays tiny.


In [ ]:
negation_rank = retrieve_query("q-negation")
numeric_rank = retrieve_query("q-numeric")
entity_rank = retrieve_query("q-entity")
print("negation", retrieval_report(negation_rank))
print("numeric", retrieval_report(numeric_rank))
print("entity", retrieval_report(entity_rank))
print("approve score under deny query", round(negation_rank.results[1].score, 4))
print("thousand score under fifty query", round(numeric_rank.results[1].score, 4))
print("4412 vs 4413 margin", round(entity_rank.margin() or 0.0, 6))
assert negation_rank.ids()[:2] == ("d-deny-refund", "d-approve-refund")
assert negation_rank.results[1].score > 0.85
assert numeric_rank.ids()[:2] == ("d-pay-fifty", "d-pay-thousand")
assert numeric_rank.results[1].score > 0.85
assert entity_rank.ids()[:2] == ("d-ticket-4412", "d-ticket-4413")
assert (entity_rank.margin() or 1.0) < 0.12


### The score does not know "not" the way a reviewer does

Deny-refund wins, and approve-refund is still above 0.85. Fifty wins,
and five thousand is still nearby. 4412 beats 4413 by a margin too
small to treat as identification. Mean-pooled embeddings often smear
negation, numerals, and neighboring ids. Report that honestly. Do not
say the vector "ignored" the user.


## Predict before running — domain shift

Timestamp a prediction before `run-domain`.

Query: `weather forecast for tomorrow`.

Predict the top document and whether legal or invoice rows can beat
rain. The encoder and corpus stay fixed; only the query domain changes.


In [ ]:
domain_rank = retrieve_query("q-domain")
print(retrieval_report(domain_rank))
for item in domain_rank.results[:5]:
    print(item.rank, item.id, round(item.score, 4), item.text)
assert domain_rank.top_id == "d-rain"
assert "d-legal" not in domain_rank.ids()[:2]
assert "d-invoice" not in domain_rank.ids()[:2]


### Domain shift is the easy win on this table

Rain is far from refunds and tickets because those axes were authored
apart. That is useful and also a reminder: a teaching table can make
domain look solved. Production encoders still fail when the query
leaves the training mixture. Do not generalize this clean split.


## Predict before running — normalization policy

Timestamp a prediction before `run-normalization`.

**Change:** keep the printer query and texts; compare

1. declared cosine on the L2 store, versus
2. inner product against **sum-pooled** unnormalized vectors.

Predict which document wins each ranking, and whether cosine on raw
means matches cosine on L2 (scale invariance).

Invariant: same token table and same sentences. Only the score
convention and pooling change.


In [ ]:
query = canonical.get("q-printer")
cosine_order = retrieve_query("q-printer")
raw_query = encoder.encode(query.text, pooling="mean", normalization="none")
ip_rows = []
for doc in canonical.documents():
    raw_mean = encoder.encode(doc.text, pooling="mean", normalization="none")
    summed = encoder.encode(doc.text, pooling="sum", normalization="none")
    ip_rows.append(
        {
            "id": doc.id,
            "cosine_l2": cosine_similarity(query.vector, doc.vector),
            "cosine_raw": cosine_similarity(raw_query, raw_mean),
            "dot_sum": inner_product(query.vector, summed),
            "n_tokens": len(content_tokens(doc.text)),
        }
    )
ip_sorted = sorted(ip_rows, key=lambda row: (-row["dot_sum"], row["id"]))
print("cosine-on-L2 winner", cosine_order.top_id)
print("dot-on-sum winner", ip_sorted[0]["id"], "tokens", ip_sorted[0]["n_tokens"])
for row in ip_rows:
    if row["id"] in {"d-printer-reset", "d-printer-queue"}:
        print(row)
        assert abs(row["cosine_l2"] - row["cosine_raw"]) < 1e-9
assert cosine_order.top_id == "d-printer-reset"
assert ip_sorted[0]["id"] == "d-printer-queue"
print("declared metric", canonical.provenance.metric, "normalization", canonical.provenance.normalization)


### Cosine is not inner product unless the store is unit

Cosine on means is invariant to scale, so L2 and raw means agree.
Inner product on **sums** grows with the number of content tokens, so
the long printer-queue document wins. A cosine index implemented as
raw dot product is only honest when both sides are L2. That is why
normalization is in the fingerprint, not a comment.


## Code reading — encode, fingerprint, score, rank, ties

Read `TeachingEncoder.encode`, `assert_compatible`, `rank_neighbors`,
and `operational_score` in `missions/M28/embedding_core.py`.

Trace, then predict before the next cell:

1. shape and L2 norm of `encode("reset my password")`
2. whether extra spaces change that vector
3. whether `rank_neighbors` on a v06.1 query and v06.2 corpus raises
   before it returns ids

Do not search the file for a score matrix over tokens or for an index
class. Stay on provenance and ranking.


In [ ]:
encode_src = inspect.getsource(encoder.encode)
rank_src = inspect.getsource(rank_neighbors)
for marker in ("mean", "l2_normalize", "lookup"):
    print("encode contains", marker, marker in encode_src)
print("rank mentions enforce_provenance", "enforce_provenance" in rank_src)
print("rank sorts by (-score, id)", "(-item.score, item.id)" in rank_src)
report = encoder.encode_report("reset my password")
print("encode_report", report)
print("spaces collapsed", encode_text("reset my password") == encode_text("reset   my   password"))
print("model", encoder.provenance.model, "version", encoder.provenance.version, "downloaded", encoder.provenance.downloaded)
print("shape", report["shape"], "norm", round(report["norm"], 6))


## Predict before running — Controlled failure: mixed provenance

Timestamp a prediction before `run-failure`.

Query: `q-password` from `v06.1` L2 cosine.
Corpus: `datasets/M28/mismatch.json` (`v06-teaching-meanpool-alt`,
`v06.2`, normalization `none`, account and print axes swapped).

The defective path sets `enforce_provenance=False` and keeps the
declared cosine metric.

Predict:
- which fingerprint fields disagree
- whether the unchecked call still returns a plausible ranking
- which class of document a password query might retrieve after an
  account/print swap

The texts stay the same. Only the store identity is wrong.


In [ ]:
query = canonical.get("q-password")
print("query fingerprint", canonical.provenance.fingerprint())
print("corpus fingerprint", mismatch.provenance.fingerprint())
print(compatibility_report(canonical.provenance, mismatch.provenance))
defective = retrieve_unchecked(
    query.vector,
    mismatch,
    query_text=query.text,
    query_id=query.id,
    query_provenance=canonical.provenance,
    top_k=5,
)
print(retrieval_report(defective))
for item in defective.results[:4]:
    print(item.rank, item.id, round(item.score, 4), item.text)
assert defective.enforced is False
assert any(item_id.startswith("d-printer") for item_id in defective.ids()[:2])
print("canonical password top", password_rank.top_id)


### Diagnose before repair

Symptom: a ranking came back with scores that look like similarities,
and a password query retrieved a printer document.

Hypotheses worth separating: the query text changed; cosine is
undefined; the stores have different fingerprints; the axis swap maps
account mass onto the print axis.

The discriminating observation is the compatibility report: `model`,
`version`, and `normalization` disagree. The axis swap explains *which*
wrong neighbor appeared. The root cause is scoring across contracts,
not a mysterious encoder.

Do not repair this by editing the query or by opening M33.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Predict that `rank_neighbors(..., enforce_provenance=True)`:

- raises `ProvenanceError` listing `model`, `version`, and
  `normalization`
- still retrieves `d-password-forgot` first on the canonical store

Do not repair by swapping axes inside the scorer.


In [ ]:
query = canonical.get("q-password")
try:
    rank_neighbors(
        query.vector,
        mismatch,
        query_text=query.text,
        query_id=query.id,
        query_provenance=canonical.provenance,
        enforce_provenance=True,
    )
    raise AssertionError("expected ProvenanceError")
except ProvenanceError as exc:
    print("raised", exc)
    print("mismatches", exc.mismatches)
    repaired_error = exc

same_store = retrieve_query("q-password")
print("same-store top", same_store.top_id, "enforced", same_store.enforced)
assert "version" in repaired_error.mismatches
assert "model" in repaired_error.mismatches
assert "normalization" in repaired_error.mismatches
assert same_store.top_id == "d-password-forgot"
print("compatibility gate restored; canonical retrieve unchanged")


### Refuse the mix; do not reinterpret it

The smallest repair is the gate. Same-store cosine still works. Logging
model, version, metric, and normalization is how M33 will fail closed
later. Re-normalizing a foreign store at query time would hide the
migration, which is the other ADR option — not this repair.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- password ranking and paraphrase comparison
- Jaccard versus cosine on printer/login and the low-overlap account pair
- negation, numeric, entity, and domain hard cases
- L2 cosine versus sum inner-product disagreement
- mixed-store diagnosis and `ProvenanceError` repair

See `missions/M28/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M28/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Use only `datasets/M28/transfer.json`. Rank by cosine, explain one
semantic success and one failure, identify the mismatch probe, and
state what an embedding score does not prove.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M28/adr_prompt.md` to choose a V06/V08 embedding
model/version/normalization/metric/compatibility policy. Do not claim a
production encoder.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M28 for the package, not as a substitute for the learner ADR.


## M15 → M27 → M28 handoff

M15 supplied metric discipline. M27 froze text as token IDs. M28
attaches offline sentence vectors and a mix-or-refuse contract.

M29 may use the idea that a vector is a representation that can change
with context. It must not relabel cosine as attention.

M33 may index these fixtures only after rankings, hard cases, and
`ProvenanceError` are defended. It must not treat `datasets/M28/` as a
production service.

Reusable artifacts: `embeddings.json`, `catalog.json`, `mismatch.json`,
`transfer.json`.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why is an embedding score a chosen metric rather than a meaning?
2. How did Jaccard and cosine disagree on `please reset the printer`?
3. Why can approve-refund still score above 0.85 under a deny query?
4. What must M33 receive that a ranking without provenance cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert password_rank.ids()[:3] == tuple(CATALOG["expected"]["q-password_top3"])
assert paraphrase_rank.ids()[:3] == tuple(CATALOG["expected"]["q-paraphrase_top3"])
assert printer_row["semantic_rank"] == 1
assert login_row["disagrees"]
assert negation_rank.results[1].score > 0.85
assert entity_rank.ids()[:2] == ("d-ticket-4412", "d-ticket-4413")
assert domain_rank.top_id == "d-rain"
assert cosine_order.top_id == "d-printer-reset"
assert ip_sorted[0]["id"] == "d-printer-queue"
assert defective.enforced is False
assert any(item_id.startswith("d-printer") for item_id in defective.ids()[:2])
assert "normalization" in repaired_error.mismatches
assert same_store.top_id == "d-password-forgot"
print("M28 integrity checks passed")
